# 14. OOP Advanced - Inheritance & MRO (5+ Years Interview Guide)
Deep architectural analysis of single and multiple inheritance, cooperative super(), Method Resolution Order (MRO), C3 Linearization, Abstract Base Classes, and classmethod factories.

### Key 5-Year Interview Concepts Covered:
- **Cooperative `super()`**: Dynamic proxy dispatching to the next class in the active MRO sequence (not hardcoded to parent).
- **C3 Linearization Algorithm**: Three invariants guaranteeing monotonicity and local precedence order in diamond hierarchies.
- **Abstract Base Classes (`abc.ABC`)**: Enforcing subclass interface contracts and polymorphic design with `@abstractmethod`.
- **Method Types**: `@classmethod` (factory constructors receiving `cls`) vs `@staticmethod` (namespace utilities) vs instance methods.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Single Class Inheritance
**Explanation**: Inheritance allows a subclass to inherit attributes and methods from a base class (`class Child(Parent):`). The subclass can use parent methods directly or extend parent behavior, fostering code reuse and polymorphic architectures.

**Syntax**: `class DerivedClass(BaseClass): ...`

In [ ]:
class BaseAuditValidator:
    def get_validator_type(self): return 'Base'
class SubAuditValidator(BaseAuditValidator):
    pass
print(SubAuditValidator().get_validator_type())

### 2. Overriding Inherited Methods
**Explanation**: A subclass can override a parent class method by defining a method with the exact same name. When invoked on an instance of the subclass, Python's attribute lookup finds the method in the subclass dictionary first, overriding the parent version.

**Syntax**: `class Child(Parent): def process(self): # Specialized logic`

In [ ]:
class BaseAuditValidator:
    def get_validator_type(self): return 'Base'
class SubAuditValidator(BaseAuditValidator):
    def get_validator_type(self): return 'Sub'
print(SubAuditValidator().get_validator_type())

### 3. Method Delegation via `super()`
**Explanation**: Instead of hardcoding the parent class name `Parent.method(self)`, use `super().method()`. `super()` returns a proxy object that delegates method calls to the next class in the object's Method Resolution Order (MRO). This makes code robust to class renaming and cooperative multiple inheritance.

**Syntax**: `super().process_data(*args, **kwargs)`

In [ ]:
class BaseAuditValidator:
    def get_validator_type(self): return 'Base'
class SubAuditValidator(BaseAuditValidator):
    def get_validator_type(self): return super().get_validator_type() + 'Sub'
print(SubAuditValidator().get_validator_type())

### 4. Calling `super().__init__()` Constructors
**Explanation**: Subclasses that implement their own `__init__` constructor must explicitly invoke `super().__init__(*args, **kwargs)` to ensure base class state is properly initialized before running subclass initialization logic.

**Syntax**: `def __init__(self, arg1, **kwargs): super().__init__(**kwargs); self.arg1 = arg1`

In [ ]:
class BaseAuditValidator:
    def __init__(self, initial_value): self.initial_value = initial_value
class SubAuditValidator(BaseAuditValidator):
    def __init__(self, initial_value, secondary_value):
        super().__init__(initial_value)
        self.secondary_value = secondary_value
print(SubAuditValidator(1, 2).initial_value, SubAuditValidator(1, 2).secondary_value)

### 5. Multiple Class Inheritance Syntax
**Explanation**: Python natively supports multiple inheritance: `class Subclass(Base1, Base2):`. The subclass inherits attributes from all listed parent classes, evaluated according to Python's C3 Linearization MRO order from left to right.

**Syntax**: `class MultiChild(MixinA, MixinB, BaseService): ...`

In [ ]:
class ParentOne: pass
class ParentTwo: pass
class ChildClass(ParentOne, ParentTwo): pass
print(ChildClass.__mro__)

### 6. Method Resolution Order (`__mro__` & `mro()`)
**Explanation**: The Method Resolution Order (MRO) defines the exact sequence of classes Python searches when resolving an attribute or method on an instance. You can inspect the MRO tuple via `ClassName.__mro__` or `ClassName.mro()`.

**Syntax**: `ClassName.__mro__` / `ClassName.mro()`

In [ ]:
class BaseClass: pass
print(BaseClass.__mro__)

### 7. The C3 Linearization Algorithm
**Explanation**: Python determines MRO using the C3 Linearization algorithm. C3 enforces three critical invariants: 1. Children precede parents; 2. Multiple parent order (left-to-right) is preserved; 3. Monotonicity (if A precedes B in one class MRO, A must precede B in all subclass MROs). If an inheritance hierarchy creates an inconsistent order, Python raises `TypeError: Cannot create a consistent method resolution order (MRO)` at class definition time.

**Syntax**: `# C3 Merge rule: L[C(B1...Bn)] = C + merge(L[B1], ..., L[Bn], B1...Bn)`

In [ ]:
class BaseClass: pass
class IntermediateClass(BaseClass): pass
class LeafClass(IntermediateClass): pass
print(LeafClass.__mro__)

### 8. Diamond Hierarchy Inheritance Paths
**Explanation**: In a diamond hierarchy (`D` inherits from `B` and `C`, both inheriting from `A`), cooperative `super()` ensures that `A.__init__` is executed EXACTLY ONCE, resolving through `D -> B -> C -> A -> object`. If classes called parent names explicitly, `A` would be called twice.

**Syntax**: `# Diamond structure: D -> B -> C -> A -> object`

In [ ]:
class DiamondTop: pass
class DiamondLeft(DiamondTop): pass
class DiamondRight(DiamondTop): pass
class DiamondBottom(DiamondLeft, DiamondRight): pass
print(DiamondBottom.__mro__)

### 9. Abstract Base Classes (`abc.ABC`)
**Explanation**: Abstract Base Classes define common interface contracts. Subclassing `abc.ABC` prevents the base class from being instantiated directly if any abstract methods remain unimplemented (`TypeError: Can't instantiate abstract class with abstract methods`).

**Syntax**: `from abc import ABC, abstractmethod; class AbstractService(ABC): ...`

In [ ]:
from abc import ABC
class AbstractBaseClass(ABC): pass
print(AbstractBaseClass)

### 10. Enforcing Interfaces with `@abstractmethod`
**Explanation**: Decorating a method with `@abstractmethod` marks it as mandatory for concrete subclasses to override. Concrete subclasses MUST implement all abstract methods before Python permits instantiation.

**Syntax**: `@abstractmethod\ndef execute_transaction(self, payload): pass`

In [ ]:
from abc import ABC, abstractmethod
class AbstractBaseClass(ABC):
    @abstractmethod
    def execute_abstract_audit(self): pass
class ConcreteSubClass(AbstractBaseClass):
    def execute_abstract_audit(self): return 'Abstract Audit Cleared'
print(ConcreteSubClass().execute_abstract_audit())

### 11. Abstract Properties (`@property` + `@abstractmethod`)
**Explanation**: Combining `@property` with `@abstractmethod` requires subclasses to implement specific read-only or read-write properties. Decorators can be stacked in Python 3.3+: `@property` above `@abstractmethod`.

**Syntax**: `@property\n@abstractmethod\ndef currency_code(self): pass`

In [ ]:
from abc import ABC, abstractmethod
class AbstractBaseClass(ABC):
    @property
    @abstractmethod
    def validated_limit(self): pass
class ConcreteSubClass(AbstractBaseClass):
    @property
    def validated_limit(self): return 10000.0
print(ConcreteSubClass().validated_limit)

### 12. Class Methods as Factory Constructors (`@classmethod`)
**Explanation**: A `@classmethod` receives the class object `cls` as its first parameter rather than an instance `self`. They are primarily used as alternative factory constructors (e.g. `from_csv_row`, `from_json`, `from_dict`) that correctly instantiate subclasses polymorphically.

**Syntax**: `@classmethod\ndef from_csv_row(cls, row): return cls(*row)`

In [ ]:
class TransactionLog:
    def __init__(self, data_value): self.data_value = data_value
    @classmethod
    def create_log_instance(cls, source_value): return cls(source_value)
print(TransactionLog.create_log_instance(100).data_value)

### 13. Static Utility Methods (`@staticmethod`)
**Explanation**: A `@staticmethod` does not receive `self` or `cls`. It behaves like a plain function placed inside the class namespace for logical grouping. Use static methods for self-contained utility calculations that do not depend on class or instance state.

**Syntax**: `@staticmethod\ndef is_valid_currency(code): return code in ('USD', 'EUR')`

In [ ]:
class TransactionLog:
    @staticmethod
    def add_amounts(amount_one, amount_two): return amount_one + amount_two
print(TransactionLog.add_amounts(200.0, 300.0))

### 14. Polymorphism & Duck Typing
**Explanation**: Python embraces 'Duck Typing' ('If it walks like a duck and quacks like a duck, it's a duck'). Polymorphic functions do not require explicit class inheritance; they only require objects to support the expected methods and interfaces at runtime.

**Syntax**: `def process_batch(payment_processors): for p in payment_processors: p.pay()`

In [ ]:
class Duck:
    def quack(self): return 'Quack'
class Person:
    def quack(self): return 'Imitates Duck'
def execute_quack_call(object_instance): print(object_instance.quack())
execute_quack_call(Duck())
execute_quack_call(Person())


### 15. Dynamic Base Class Inspection (`__bases__` & `__subclasses__`)
**Explanation**: `ClassName.__bases__` returns a tuple of direct parent classes. `ClassName.__subclasses__()` returns a list of all active subclasses registered in memory. This is widely used in plugin architectures to auto-discover installed extensions.

**Syntax**: `all_subclasses = BasePlugin.__subclasses__()`

In [ ]:
class BaseOne:
    def get_base_name(self): return 'BaseOne'
class BaseTwo:
    def get_base_name(self): return 'BaseTwo'
class ConcreteClass(BaseOne): pass
ConcreteClass.__bases__ = (BaseTwo,)
print('Dynamic Base reassigned:', ConcreteClass().get_base_name())


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Building polymorphic financial auditing frameworks with Abstract Base Classes, classmethod factory loaders, and cooperative MRO inheritance.


In [ ]:
# Solution:
from abc import ABC, abstractmethod
class AbstractTransaction(ABC):
    @abstractmethod
    def audit_status(self): pass

class AuditLog(AbstractTransaction):
    def audit_status(self): return 'CLEARED'
    
print('Audited Status:', AuditLog().audit_status())


### Q2: Classmethod Factory Loader for CSV Streaming
**Explanation**: **Scenario**: Implement a `@classmethod` factory `from_csv_line` on a `TransactionRecord` subclass to parse raw CSV lines directly into validated domain model instances.

**Syntax**: `record = TransactionRecord.from_csv_line(raw_line)`

In [ ]:
# Solution:
class AutoTx:
    def __init__(self, tx_id, amount):
        self.tx_id = tx_id
        self.amount = amount
    @classmethod
    def from_csv_row(cls, row):
        return cls(row[0], float(row[3]))

row_data = ['TX999', 'C10', 'M20', '500.00']
tx = AutoTx.from_csv_row(row_data)
print('Factory loaded ID:', tx.tx_id, 'Amount:', tx.amount)
